# CS 513 Final Project - NYPD Vehicle Stop Report Data
## Data Cleaning

Name: Samuel Preston and Dean Filippone  
CWID: 10463953 and 20006018  
Assignment: Final Project  
Purpose: Analyze data regarding NYPD Traffic Stops  

##### Imports

In [29]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

#models imports
# Naive Bayes
from sklearn.naive_bayes import GaussianNB, CategoricalNB

# ANN (Artificial Neural Network)
from sklearn.neural_network import MLPClassifier

# KNN (K-Nearest Neighbors)
from sklearn.neighbors import KNeighborsClassifier

# CART (Classification and Regression Trees)
from sklearn.tree import DecisionTreeClassifier

# Logistic Regression
from sklearn.linear_model import LogisticRegression

# Random Forest
from sklearn.ensemble import RandomForestClassifier

# SVM
from sklearn.svm import SVC

# Gradient Boosting
from sklearn.ensemble import GradientBoostingClassifier

# XGBoost
from xgboost import XGBClassifier

##### Reading Data

In [30]:
df = pd.read_csv('NYPD_Vehicle_Stop_Reports_20250430.csv')

## Data Manipulation & Cleaning

#### Relabeling Data

In [31]:
df_cleaned = df.copy()

df_cleaned = df_cleaned.rename(columns={
    'EVNT_KEY': 'event_id',
    'OCCUR_DT': 'stop_date',
    'OCCUR_TM': 'stop_time',
    'CMD_CD': 'command_code',
    'VEH_SEIZED_FLG': 'vehicle_seized',
    'VEH_SEARCHED_FLG': 'vehicle_searched',
    'VEH_SEARCH_CONSENT_FLG': 'search_consent',
    'VEH_CHECKPOINT_FLG': 'checkpoint_stop',
    'FORCE_USED_FLG': 'force_used',
    'ARREST_MADE_FLG': 'arrest_made',
    'SUMMON_ISSUED_FLG': 'summons_issued',
    'VEH_CATEGORY': 'vehicle_type',
    'RPTED_AGE': 'driver_age',
    'SEX_CD': 'driver_sex',
    'RACE_DESC': 'driver_race',
    'LATITUDE': 'latitude',
    'LONGITUDE': 'longitude',
    'X_COORD_CD': 'x_coord',
    'Y_COORD_CD': 'y_coord',
    'datetime': 'stop_datetime'
})

#### Creating DateTime, Stop DayOfWeek, Stop Month, Stop Year and Stop Season columns

In [32]:
# Combine stop_date and stop_time into a datetime column
df_cleaned['stop_datetime'] = pd.to_datetime(df_cleaned['stop_date'] + ' ' + df_cleaned['stop_time'], errors='coerce')

# Extract parts of the date
df_cleaned['stop_dayofweek'] = df_cleaned['stop_datetime'].dt.day_name()
df_cleaned['stop_month'] = df_cleaned['stop_datetime'].dt.month
df_cleaned['stop_year'] = df_cleaned['stop_datetime'].dt.year

# Map months to seasons (Northern Hemisphere)
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

df_cleaned['stop_season'] = df_cleaned['stop_month'].apply(get_season)

#### Creating Stop Time of Day Column

In [33]:
# Extract hour from stop_datetime
df_cleaned['stop_hour'] = df_cleaned['stop_datetime'].dt.hour

# Categorize time of day
def get_time_of_day(hour):
    if hour >= 0 and hour < 6:
        return 'Late Night'
    elif hour < 12:
        return 'Morning'
    elif hour < 18:
        return 'Afternoon'
    else:
        return 'Evening'

df_cleaned['stop_time_of_day'] = df_cleaned['stop_hour'].apply(get_time_of_day)

#### Cleaning Command Code attribute

In [34]:
# Convert command_code to string type
df_cleaned['command_code'] = df_cleaned['command_code'].astype(str)

#### Cleaning Search Consent Attribute

In [35]:
df_cleaned['search_consent'] = df_cleaned['search_consent'].replace({'(null)': None})
df_cleaned['search_consent'] = df_cleaned['search_consent'].map({
    'Y': 'CONSENTED',
    'N': 'DENIED'
}).fillna('UNKNOWN')

#### Cleaning Vehicle Type Column

In [36]:
df_cleaned['vehicle_type'] = df_cleaned['vehicle_type'].replace({'(null)': 'UNKNOWN'})
df_cleaned['vehicle_type'].unique()

array(['CAR/SUV', 'TLC', 'TRUCK/BUS', 'MCL', 'OTHER', 'BIKE', 'UNKNOWN'],
      dtype=object)

#### Cleaning Driver Age column

In [37]:
df_cleaned['driver_age'] = pd.to_numeric(df_cleaned['driver_age'], errors='coerce')
df_cleaned = df_cleaned[(df_cleaned['driver_age'] >= 1) & (df_cleaned['driver_age'] <= 120)]

#### Cleaning Driver Sex column

In [38]:
df_cleaned['driver_sex'] = df_cleaned['driver_sex'].where(df_cleaned['driver_sex'].isin(['M', 'F']), 'UNKNOWN')

#### Cleaning Driver Race Column

In [39]:
df_cleaned['driver_race'] = df_cleaned['driver_race'].replace({'(null)': 'UNKNOWN'})
df_cleaned['driver_race'] = df_cleaned['driver_race'].replace({
    '(null)': 'UNKNOWN',
    'BLACK HISPANIC': 'HISPANIC',
    'WHITE HISPANIC': 'HISPANIC',
    'ASIAN / PACIFIC ISLANDER': 'ASIAN',
    'AMERICAN INDIAN/ALASKAN NATIVE': 'NATIVE'
})

#### Cleaning Location Data

In [40]:
df_cleaned = df_cleaned[
    (df_cleaned['latitude'].notnull() & df_cleaned['longitude'].notnull()) |
    (df_cleaned['x_coord'].notnull() & df_cleaned['y_coord'].notnull())
]

#### Dropping Search Consented

In [41]:
df_cleaned = df_cleaned.drop(columns=['search_consent'])

#### Showing Dataframe

In [42]:
df_cleaned

,event_id,stop_date,stop_time,command_code,vehicle_seized,vehicle_searched,checkpoint_stop,force_used,arrest_made,summons_issued,...,longitude,x_coord,y_coord,stop_datetime,stop_dayofweek,stop_month,stop_year,stop_season,stop_hour,stop_time_of_day
0,284650033,04/02/2024,07:04:00,1,False,False,False,False,False,True,...,-74.005913,982611.0,202413.0,2024-04-02 07:04:00,Tuesday,4,2024,Spring,7,Morning
1,284650036,04/02/2024,08:56:00,5,False,False,False,False,False,True,...,-73.994809,985689.0,201030.0,2024-04-02 08:56:00,Tuesday,4,2024,Spring,8,Morning
2,284650037,04/02/2024,07:48:00,5,False,False,False,False,False,True,...,-73.994809,985689.0,201030.0,2024-04-02 07:48:00,Tuesday,4,2024,Spring,7,Morning
3,284650038,04/02/2024,04:55:00,7,False,False,False,False,False,False,...,-73.986721,987931.0,200118.0,2024-04-02 04:55:00,Tuesday,4,2024,Spring,4,Late Night
4,284650039,04/02/2024,07:10:00,7,False,False,False,False,False,False,...,-73.986721,987931.0,200118.0,2024-04-02 07:10:00,Tuesday,4,2024,Spring,7,Morning
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
677663,304404459,03/31/2025,16:15:00,83,False,False,False,False,False,True,...,0.000000,1008705.0,193427.0,2025-03-31 16:15:00,Monday,3,2025,Spring,16,Afternoon
677664,304404690,03/31/2025,9:00:00,109,False,False,False,False,False,False,...,0.000000,1031437.0,216900.0,2025-03-31 09:00:00,Monday,3,2025,Spring,9,Morning
677665,304404735,03/31/2025,17:30:00,110,False,False,False,False,False,False,...,-73.844416,1027233.0,215480.0,2025-03-31 17:30:00,Monday,3,2025,Spring,17,Afternoon
677666,304424562,03/31/2025,23:24:00,47,False,False,False,False,True,False,...,-73.860813,1022790.0,264806.0,2025-03-31 23:24:00,Monday,3,2025,Spring,23,Evening


### Printing to a new CSV

In [43]:
df_cleaned.to_csv("nypd_vehicle_stops_cleaned.csv", index=False)